In [1]:
# Auto-generated by build_and_verify.py — do not hand-edit.
# Rebuild with: python build_and_verify.py lesson_05_pbc_and_neighbor_lists.yaml

from cs470_engine import Worksheet
from cs470_engine.plot_style import apply_default_style

apply_default_style()

ws = Worksheet.load('lesson_05_pbc_and_neighbor_lists.yaml')
ws.start()


$\def\AA{\text{Å}}$

# MSE 485 · Lesson 05: Spatial Boundary Conditions, Minimum Image Convention & Neighbor Lists

_Session 9f4cb280 · Estimated time: ~95 minutes · 38 problems_

**Recording your answers.** Submit each problem as you go. Your answers are saved to the workspace as you submit, *and* the running record at the bottom of the notebook keeps your progress. When you finish, run the last two cells — **`show_record()`** and **`finalize()`** — to record your participation. **If you reopened this workspace** (or the kernel restarted), use **Kernel → Restart Kernel and Run All Cells** before finalizing, so every answer is reloaded.

### Learning Objectives
- Quantify the surface-to-volume scaling ($\propto N^{-1/3}$) of finite 3D atomic clusters and explain why Periodic Boundary Conditions (PBC) are essential for simulating bulk phases.
- Identify physical artifacts that persist under PBC—including long-wavelength phonon cutoff ($\lambda > L$), periodic self-interaction, nucleation suppression, and cubic box anisotropy.
- Apply coordinate wrapping and the Minimum Image Convention (MIC) in 1D and 3D, and prove why the radial cutoff must strictly satisfy $r_c < L/2$.
- Compare potential truncation schemes (raw cutoff, shifted potential, and shifted-force potential) and their effects on energy conservation in Molecular Dynamics.
- Analyze the $\mathcal{O}(N^2)$ bottleneck of all-pairs distance checking and reduce force evaluations to $\mathcal{O}(N)$ using Verlet skin lists ($r_v = r_c + \Delta$, rebuild trigger $d_1 + d_2 \ge \Delta$) and $\mathcal{O}(N)$ Cell Linked-Lists (`head` and `list` arrays).


In [2]:
ws.render_section('sec_surface_pbc_intro')

## 1. Spatial Boundary Conditions & The Tyranny of Surfaces

Macroscopic materials contain $\mathcal{O}(10^{23})$ atoms, whereas atomistic Molecular Dynamics (MD) and Monte Carlo (MC) simulations typically track $N \sim 10^3\text{–}10^7$ atoms. Because spatial scales are limited, our choice of **spatial boundary conditions** fundamentally determines whether the simulation represents a nanoscale cluster or a macroscopic bulk phase.

### Free Boundaries vs. Bulk Scaling
In a 3D cluster of $N = n^3$ atoms with **free boundaries** (e.g., an isolated droplet or nanoparticle in vacuum), any atom within a surface shell of thickness $d$ atomic layers experiences an under-coordinated, non-bulk environment. The number of strictly interior (bulk) atoms is:
$$N_{\text{bulk}} = (n - 2d)^3 = \left(N^{1/3} - 2d\right)^3$$
Expanding for large $N$, the fraction of atoms residing in the surface layer scales slowly as an inverse cube root:
$$\frac{N_{\text{surf}}}{N} = 1 - \left(1 - \frac{2d}{N^{1/3}}\right)^3 \approx \frac{6d}{N^{1/3}} \propto N^{-1/3}$$
- For a $N = 10^3 = 1000$ atom cube with a 1-atom surface layer ($d=1$), $N_{\text{bulk}} = 8^3 = 512$, leaving **$488$ atoms ($48.8\%$) on the surface**!
- Even for $N = 10^6 = (100)^3$ atoms with a $d=5$ atom thick surface interaction region, $1 - (90/100)^3 = 27.1\%$ (roughly **a quarter of all atoms**) are still dominated by surface forces!

### Types of Boundary Conditions
1. **Free boundaries:** Isolated cluster, droplet, or protein in vacuum.
2. **Periodic Boundary Conditions (PBC):** The primary simulation box of side $L$ is replicated infinitely in all directions. When a particle leaves through one face, its periodic image simultaneously enters through the opposite face. PBC **completely eliminates free surfaces**, allowing small $N$ to probe bulk solids, liquids, gases, and plasmas.
3. **Mixed (Slab) boundaries:** Periodic in $x$ and $y$, but free (or reflective/wall-bounded) in $z$—ideal for studying free surfaces, thin films, or solid-liquid interfaces.
4. **Coupled / Embedded boundaries:** Embedding an atomistic quantum or MD region inside a continuum elastic or coarse-grained bath via boundary forces and displacements.

### Remaining Finite-Size Artifacts Under PBC
Although PBC removes free surfaces, the system is still periodic with period $L$, NOT truly infinite:
- **Long-wavelength cutoff ($\lambda > L$):** Any collective excitation (phonon, acoustic wave, or capillary wave) must be commensurate with the box: $\mathbf{k} = \frac{2\pi}{L}(n_x, n_y, n_z)$, suppressing wavelengths $\lambda > L$.
- **Periodic image self-interaction:** A defect (dislocation, vacancy cluster, or charged ion) interacts elastically or electrostatically with its own periodic images separated by $L$.
- **Suppressed critical fluctuations & nucleation:** Phase transitions or critical fluctuations whose correlation length $\xi \gtrsim L$ are artificially stabilized or suppressed.
- **Cubic box anisotropy:** The periodic lattice of images has cubic symmetry rather than true isotropic rotational invariance.


In [3]:
ws.render_section('concept_surface_scaling')

### Interactive Explorer: Surface Atom Fraction in Finite 3D Clusters

Adjust the cubic cluster side length $n$ ($N = n^3$ atoms) and the surface interaction shell thickness $d$ (in atomic layers) to see why free boundaries cannot approximate bulk behavior without millions of atoms.

**Key Takeaway:** Because surface fraction decays only as $N^{-1/3}$, eliminating surface artifacts by brute-force cluster size is computationally prohibitive. Periodic Boundary Conditions (PBC) eliminate surfaces entirely at any $N$.

In [4]:
ws.render_section('q01_surface_fraction_1000')

**Q1 · easy** · 💡

Consider a 3D simple cubic nanocrystal containing $N = 1000$ atoms arranged as a $10 \times 10 \times 10$ cube in vacuum (free boundary conditions). If the outermost single layer of atoms ($d = 1$) on all six faces is considered the surface shell, what **percentage** of the atoms in this cluster are surface atoms? (Enter a number between 0 and 100, e.g., `48.8`.)


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Count the interior bulk atoms first: removing 1 layer from each end of the 3 axes leaves an $(10 - 2) \times (10 - 2) \times (10 - 2) = 8^3$ interior cube.

</div>
</details>

_Numeric answer required._

In [5]:
ws.render_section('q02_surface_fraction_million')

**Q2 · medium** · 💡

As discussed in the lecture notes, suppose a simulated droplet has $N = 10^6 = (100)^3$ atoms on a cubic grid, and the surface region affected by under-coordination is $d = 5$ atomic layers thick on every face. Calculate the **exact percentage** of atoms that lie within this 5-atom-thick surface region.


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

With $n = 100$ and $d = 5$ on both sides of each axis, the unaffected interior cube has side length $n - 2d = 100 - 10 = 90$.

</div>
</details>

_Numeric answer required._

In [6]:
ws.render_section('q03_surface_scaling_exponent')

**Q3 · easy** · 💡

In a $D$-dimensional cluster of $N$ particles with free boundaries, the surface-to-volume ratio (fraction of atoms near the surface) scales asymptotically with particle count $N$ as:


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Linear dimension $L \propto N^{1/D}$, volume $\propto L^D$, and surface area $\propto L^{D-1}$.

</div>
</details>

- [ ] **(A)** $N^{-1/3}$ in 3D (and $N^{-1/D}$ in $D$ dimensions)
- [ ] **(B)** $N^{-1}$ in all dimensions
- [ ] **(C)** $N^{-2/3}$ in 3D
- [ ] **(D)** $N^{-1/2}$ in 3D

In [7]:
ws.render_section('q04_pbc_purpose')

**Q4 · easy** · 💡

What is the primary physical motivation for applying 3D Periodic Boundary Conditions (PBC) in an atomistic simulation of $N = 4000$ liquid Argon atoms?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider what happens to atoms near the boundary of the primary simulation cell.

</div>
</details>

- [ ] **(A)** To eliminate free surfaces so that every atom experiences a bulk-like local environment despite small $N$
- [ ] **(B)** To make the total number of degrees of freedom strictly infinite in computer memory
- [ ] **(C)** To force all interatomic forces to vanish beyond $r = L$
- [ ] **(D)** To prevent atoms from ever crossing the walls of the primary simulation box

In [8]:
ws.render_section('q05_mixed_boundary_selection')

**Q5 · easy** · 💡

You want to use Molecular Dynamics to measure the surface tension and inward surface relaxation of a Copper $(001)$ crystal facet exposed to vacuum. Which spatial boundary condition setup is most appropriate?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

You want an infinite macroscopic crystal plane along $x$ and $y$, but an exposed free surface normal to $z$.

</div>
</details>

- [ ] **(A)** Mixed (slab) boundaries: periodic in $x$ and $y$, free boundaries (vacuum gap) in $z$
- [ ] **(B)** 3D Periodic Boundary Conditions with zero vacuum gap in $x$, $y$, and $z$
- [ ] **(C)** Free boundary conditions in all three directions ($x$, $y$, and $z$)
- [ ] **(D)** Spherical boundary conditions on a 4D hypersphere

In [9]:
ws.render_section('q06_pbc_remaining_artifacts')

**Q6 · medium** · 💡

Even though 3D Periodic Boundary Conditions eliminate free surfaces, finite-size artifacts remain in a cubic simulation box of side length $L$. Select **all** true finite-size artifacts that can occur under 3D PBC:


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Think about acoustic wavelengths, defect fields, and critical fluctuations relative to box length $L$.

</div>
</details>

- [ ] **(A)** Phonons and capillary waves with wavelengths $\lambda > L$ are suppressed because wavevectors must satisfy $\mathbf{k} = \frac{2\pi}{L}(n_x, n_y, n_z)$
- [ ] **(B)** A point defect, dislocation, or charged ion can interact spuriously with its own periodic images across the box length $L$
- [ ] **(C)** Phase nucleation or critical fluctuations with correlation length $\xi > L$ cannot form properly
- [ ] **(D)** Approximately $49\%$ of the atoms in a $N=1000$ periodic box lose half their coordination partners at the box faces

In [10]:
ws.render_section('q07_max_phonon_wavelength')

**Q7 · medium** · 💡

A 3D periodic simulation cell of crystalline Silicon has cubic side length $L = 5.43\text{ nm}$. What is the **longest non-zero acoustic phonon wavelength** $\lambda_{\max}$ (in $\text{nm}$) that can propagate parallel to the $[100]$ axis ($x$-axis) in this simulation?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Periodic boundary conditions require $u(x + L) = u(x)$, so an integer number of wavelengths must fit inside $L$: $n \lambda = L$.

</div>
</details>

_Numeric answer required._

In [11]:
ws.render_section('sec_mic_and_cutoffs')

## 2. Coordinate Wrapping, Minimum Image Convention & Potential Truncation

In a periodic cubic box of side length $L$ spanning $[0, L)^3$, we must distinguish between two fundamental operations: **wrapping particle coordinates** back into the primary box, and computing the **Minimum Image Convention (MIC) displacement vector** between a pair of particles.

### 1. Coordinate Wrapping (Keeping Particles in $[0, L)$)
When a particle moves outside $[0, L)$ during an integration step, its position is wrapped back into the primary cell using the **floor** function (or modulo operator `%` in Python):
$$x_{\text{wrapped}} = x - L \lfloor x / L \rfloor \quad \in [0, L)$$

### 2. Minimum Image Convention (MIC) for Pair Displacements
To compute the force between particle $i$ at $\mathbf{r}_i$ and particle $j$ at $\mathbf{r}_j$, particle $i$ must interact only with the **single closest periodic image** of particle $j$:
$$|\mathbf{r}_{ij}|_{\text{M}} = \min_{\mathbf{n} \in \mathbb{Z}^3} |\mathbf{r}_i - \mathbf{r}_j + \mathbf{n}L|$$
Component-wise along each axis $d \in \{x, y, z\}$, with $\Delta x = x_i - x_j$:
$$\Delta x_{\text{mic}} = \Delta x - L \cdot \operatorname{nint}\!\left(\frac{\Delta x}{L}\right) \quad \in \left[-\frac{L}{2}, +\frac{L}{2}\right]$$
where $\operatorname{nint}(u)$ rounds $u$ to the **nearest integer**. If $x_i, x_j \in [0, L)$ are already wrapped, this is equivalently implemented with conditional shifts:
```python
dx = x[i] - x[j]
for d in range(3):
    if dx[d] >  L[d] / 2: dx[d] -= L[d]
    if dx[d] < -L[d] / 2: dx[d] += L[d]
```

### 3. The Strict Cutoff Condition: $r_c < L/2$
For a spherically truncated short-range potential ($V_c(r) = 0$ for $r > r_c$), the cutoff radius $r_c$ must be **strictly less than half the box length**:
$$r_c < \frac{L}{2}$$
Why? The region of space closer to $\mathbf{r}_i$ than to any of its periodic images is a cube of side $L$ centered at $\mathbf{r}_i$, spanning $[-L/2, +L/2]^3$. The largest sphere inscribed inside this cube has radius $L/2$. If $r_c \ge L/2$, the cutoff sphere extends outside the minimum-image cube, allowing particle $i$ to interact simultaneously with **two or more periodic images of the same particle $j$** (or causing discontinuous jumps in force as a particle crosses the $\pm L/2$ midplane).

### 4. Handling the Tail of the Potential at $r = r_c$
Simply setting $V(r) = 0$ for $r > r_c$ creates a step discontinuity $\Delta V = -V(r_c)$ at $r = r_c$, which corresponds to an infinite delta-function impulse force $-\nabla V$ whenever a pair crosses $r_c$, destroying energy conservation in NVE MD! Three standard remedies exist:
1. **Shifted Potential (Continuous $V$, Discontinuous $F$):**
   $$V_c(r) = V(r) - V(r_c) \quad (r \le r_c)$$
   Removes the jump in potential energy at $r_c$, though the force $F(r_c) = -V'(r_c) \ne 0$ still has a small jump.
2. **Shifted-Force Potential (Continuous $V$ and Continuous $F$):**
   $$V_{\text{sf}}(r) = V(r) - V(r_c) - \left.\frac{dV}{dr}\right|_{r_c}(r - r_c) \quad (r \le r_c)$$
   Both $V_{\text{sf}}(r_c) = 0$ and $F_{\text{sf}}(r_c) = 0$ vanish smoothly at the cutoff, yielding excellent MD energy conservation.
3. **Long-Range Image Potential (Ewald Summation):**
   For potentials that decay as $r^{-\alpha}$ with $\alpha \le d$ (such as Coulomb $1/r$ in 3D), the tail integral $\int_{r_c}^\infty 4\pi r^2 V(r)\,dr$ diverges! One cannot truncate Coulomb interactions; instead, one sums over all periodic images $V_{\text{image}}(\mathbf{r}_{ij}) = \sum_{\mathbf{n}} V(\mathbf{r}_i - \mathbf{r}_j + \mathbf{n}L)$ using **Ewald summation** ($\mathcal{O}(N^{3/2})$ or $\mathcal{O}(N\log N)$ with PME) or the **Fast Multipole Method** ($\mathcal{O}(N)$).


In [12]:
ws.render_section('concept_minimum_image')

### Interactive Explorer: Minimum Image Convention & The $r_c < L/2$ Limit

Move target particle $j$'s coordinates $(x_j, y_j)$ inside the primary box $[0, L]^2$ and vary the cutoff radius $r_c / L$. Compare the dashed gray primary-box vector $\Delta\mathbf{r}_{\mathrm{raw}}$ against the green Minimum Image vector $\Delta\mathbf{r}_{\mathrm{mic}}$, and observe what happens when $r_c \ge 0.50L$.

**Key Takeaway:** Whenever $|x_i - x_j| > L/2$, the closest image of particle $j$ lives in an adjacent periodic tile rather than the primary box. Keeping $r_c < L/2$ guarantees that the cutoff circle stays strictly inside the yellow $L \times L$ minimum-image square.

In [13]:
ws.render_section('q08_wrap_vs_mic_formula')

**Q8 · easy** · 💡

In a 1D periodic box of length $L$ spanning $[0, L)$, which pair of formulas correctly distinguishes **wrapping a coordinate $x$ into $[0, L)$** from **computing the Minimum Image displacement $\Delta x_{\text{mic}} \in [-L/2, L/2]$**?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Coordinate wrapping maps into $[0, L)$ using `floor`, whereas minimum-image displacement maps into $[-L/2, L/2]$ using nearest integer `nint`/`round`.

</div>
</details>

- [ ] **(A)** Wrap: $x \leftarrow x - L\lfloor x/L \rfloor$; Minimum Image: $\Delta x \leftarrow \Delta x - L\operatorname{nint}(\Delta x/L)$
- [ ] **(B)** Wrap: $x \leftarrow x - L\operatorname{nint}(x/L)$; Minimum Image: $\Delta x \leftarrow \Delta x - L\lfloor \Delta x/L \rfloor$
- [ ] **(C)** Both operations use $u \leftarrow u - L\lfloor u/L \rfloor$
- [ ] **(D)** Both operations use $u \leftarrow u - L\lceil u/L \rceil$

In [14]:
ws.render_section('q09_coordinate_wrapping_calc')

**Q9 · easy** · 💡

A cubic simulation box has side length $L = 10.0\text{ Å}$ spanning $[0, 10.0)\text{ Å}$. After a Verlet position update, an atom's $x$-coordinate is $x = -2.3\text{ Å}$. What is its wrapped $x$-coordinate (in $\text{Å}$) inside the primary box $[0, L)$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Apply $x_{\text{wrapped}} = x - L \lfloor x/L \rfloor$ with $\lfloor -0.23 \rfloor = -1$.

</div>
</details>

_Numeric answer required._

In [15]:
ws.render_section('q10_mic_1d_displacement')

**Q10 · easy** · 💡

In a 1D periodic box of length $L = 10.0\text{ Å}$, particle $i$ is at $x_i = 1.2\text{ Å}$ and particle $j$ is at $x_j = 8.9\text{ Å}$. What is the signed Minimum Image displacement $\Delta x_{\text{mic}} = (x_i - x_j)_{\text{mic}}$ in $\text{Å}$? (Include the correct sign!)


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

First compute $\Delta x = x_i - x_j = 1.2 - 8.9 = -7.7\text{ Å}$. Since $\Delta x < -L/2$, add $L$.

</div>
</details>

_Numeric answer required._

In [16]:
ws.render_section('q11_mic_3d_distance')

**Q11 · medium** · 💡

In a 3D cubic periodic box of side length $L = 10.0\text{ Å}$, atom $A$ is at $\mathbf{r}_A = (1.0, 9.0, 2.0)\text{ Å}$ and atom $B$ is at $\mathbf{r}_B = (8.0, 2.0, 6.0)\text{ Å}$. Calculate the Minimum Image distance $|\mathbf{r}_{AB}|_{\text{M}}$ in $\text{Å}$.


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Find the minimum-image displacement along each axis first: $\Delta x_{\text{mic}}, \Delta y_{\text{mic}}, \Delta z_{\text{mic}} \in [-5, 5]$, then take the Euclidean norm.

</div>
</details>

_Numeric answer required._

In [17]:
ws.render_section('q12_cutoff_half_box_limit')

**Q12 · medium** · 💡

Why must the spherical potential cutoff radius $r_c$ strictly satisfy $r_c < L/2$ when using the Minimum Image Convention in a cubic periodic box of side length $L$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look at the interactive explorer when $r_c > 0.5L$: how many images of particle $j$ can fall inside $r_c$?

</div>
</details>

- [ ] **(A)** If $r_c \ge L/2$, a particle's cutoff sphere extends beyond the $[-L/2, L/2]^3$ Wigner-Seitz cell and can include multiple periodic images of the same neighbor
- [ ] **(B)** If $r_c \ge L/2$, the Lennard-Jones potential becomes repulsive at all distances
- [ ] **(C)** If $r_c \ge L/2$, coordinates can no longer be wrapped into $[0, L)$ using `floor`
- [ ] **(D)** If $r_c \ge L/2$, Newton's third law $\mathbf{F}_{ij} = -\mathbf{F}_{ji}$ is violated inside $r < L/4$

In [18]:
ws.render_section('q13_min_fcc_atoms_for_cutoff')

**Q13 · hard** · 💡

You are setting up a 3D periodic FCC Argon crystal with conventional cubic lattice constant $a = 1.55\,\sigma$ ($4$ atoms per unit cell) and a Lennard-Jones cutoff of $r_c = 2.5\,\sigma$. To strictly satisfy $r_c < L/2$ for an $M \times M \times M$ supercell of FCC unit cells (where $L = M a$), what is the **minimum total number of atoms $N = 4M^3$** required?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

First find the minimum integer $M$ such that $L = M a > 2 r_c = 5.0\,\sigma$, then compute $N = 4M^3$.

</div>
</details>

_Numeric answer required._

In [19]:
ws.render_section('q14_max_mic_corner_distance')

**Q14 · medium** · 💡

In a 3D cubic periodic box of side length $L$, what is the **maximum possible Minimum Image distance** $|\mathbf{r}_{ij}|_{\text{M}}$ between any two points in the box?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Each component $\Delta x_{\text{mic}}, \Delta y_{\text{mic}}, \Delta z_{\text{mic}}$ can be as large in magnitude as $L/2$.

</div>
</details>

- [ ] **(A)** $\frac{\sqrt{3}}{2} L \approx 0.866 L$
- [ ] **(B)** $\frac{1}{2} L = 0.500 L$
- [ ] **(C)** $\frac{\sqrt{2}}{2} L \approx 0.707 L$
- [ ] **(D)** $\sqrt{3} L \approx 1.732 L$

In [20]:
ws.render_section('q15_potential_truncation_schemes')

**Q15 · medium** · 💡

In the lecture notes on handling the tail of a short-range potential at $r = r_c$, the **continuous potential and continuous force** (shifted-force) modification for $r \le r_c$ is defined as:


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Subtract a first-order Taylor linear term at $r_c$ so both $V_c(r_c) = 0$ and $V_c'(r_c) = 0$.

</div>
</details>

- [ ] **(A)** $V_c(r) = V(r) - V(r_c) - \left.\frac{dV}{dr}\right|_{r_c}(r - r_c)$
- [ ] **(B)** $V_c(r) = V(r) - V(r_c)$
- [ ] **(C)** $V_c(r) = V(r) + \left.\frac{dV}{dr}\right|_{r_c} r$
- [ ] **(D)** $V_c(r) = V(r) \cdot \exp(-r / r_c)$

In [21]:
ws.render_section('q16_lj_shift_value')

**Q16 · medium** · 💡

For a Lennard-Jones 12-6 potential $V(r) = 4\varepsilon\left[(\sigma/r)^{12} - (\sigma/r)^6\right]$ truncated and shifted at $r_c = 2.5\,\sigma$ so that $V_c(r) = V(r) - V(r_c)$, what is the value of the potential energy at the cutoff prior to shifting, $V(r_c)/\varepsilon$? (Include the negative sign, e.g., `-0.0163`.)


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $4 \times (2.5^{-12} - 2.5^{-6})$.

</div>
</details>

_Numeric answer required._

In [22]:
ws.render_section('q17_force_loop_code_analysis')

**Q17 · medium** · 💡

Inspect the Lennard-Jones force loop from the lecture slides:
```python
for i in range(Natoms):
    for j in range(i+1, Natoms):
        dx = x[i] - x[j]
        for d in range(3):
            if dx[d] >  L[d]/2: dx[d] -= L[d]
            if dx[d] < -L[d]/2: dx[d] += L[d]
        r2 = sum(dx*dx)
        if r2 > rcutoff2: continue
        ...
        F[i] = F[i] + rforce * dx
        F[j] = F[j] - rforce * dx
```
Select **all** true statements about the computational design of this loop:


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look at the loop bounds `j in range(i+1, Natoms)`, the `r2 > rcutoff2` check, and the updates to `F[i]` and `F[j]`.

</div>
</details>

- [ ] **(A)** Starting the inner loop at `j = i + 1` and updating both `F[i]` and `F[j]` exploits Newton's 3rd Law ($\mathbf{F}_{ij} = -\mathbf{F}_{ji}$), halving the number of pair evaluations to $N(N-1)/2$
- [ ] **(B)** Comparing squared distances `r2 > rcutoff2` avoids computing an expensive square root `sqrt(r2)` for every pair
- [ ] **(C)** Before entering the outer `for i` loop at each MD step, the force array and potential energy must be zeroed (`F.fill(0)`, `pot = 0`)
- [ ] **(D)** Because of the `if r2 > rcutoff2: continue` statement, the total complexity of this nested loop is $\mathcal{O}(N)$

In [23]:
ws.render_section('q18_long_range_potentials_complexity')

**Q18 · medium** · 💡

Why can't a Coulomb interaction $V(r) = q_i q_j / r$ in 3D simply be truncated at $r_c < L/2$ like a Lennard-Jones potential, and what is the scaling of standard Ewald summation vs. the Fast Multipole Method (FMM)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

In 3D, a shell at radius $r$ has volume $4\pi r^2 dr$, so the tail integral $\int r^2 (1/r) dr$ diverges; check slide 6 of `05.PBC-neighbors.pdf` for Ewald and FMM scaling.

</div>
</details>

- [ ] **(A)** The $1/r$ tail integral diverges in 3D; standard Ewald summation scales as $\mathcal{O}(N^{3/2})$ while Fast Multipole Algorithms scale as $\mathcal{O}(N)$ for very large $N$
- [ ] **(B)** The $1/r$ potential is short-ranged; Ewald summation scales as $\mathcal{O}(N^3)$ and FMM scales as $\mathcal{O}(N^2)$
- [ ] **(C)** Coulomb forces violate Newton's third law; Ewald summation scales as $\mathcal{O}(\log N)$
- [ ] **(D)** Truncation works accurately for $1/r$ as long as $r_c = 2.5\,\sigma$; Ewald summation is only used for hard spheres

In [24]:
ws.render_section('sec_verlet_neighbor_lists')

### The $\mathcal{O}(N^2)$ All-Pairs Bottleneck
In a system of $N$ atoms with a short-range potential ($V_c(r) = 0$ for $r > r_c$), each atom interacts on average with only:
$$m \approx \frac{4}{3}\pi r_c^3 \rho$$
neighbors inside its cutoff sphere, where $\rho = N/V$ is the number density. Crucially, **$m$ is independent of system size $N$** (typically $m \sim 30\text{–}80$ in a liquid or solid). Yet the naive double loop checks all $\frac{1}{2}N(N-1) \approx \mathcal{O}(N^2)$ pairs just to discover that almost all of them have $r_{ij} > r_c$!

### Verlet Neighbor Lists (Explicit Neighbor Tables)
Introduced by Loup Verlet (1967), a **Verlet neighbor list** stores a table of nearby neighbor indices for each atom $i$.
- Instead of cutting off the table at $r_c$, we construct the neighbor table using a slightly larger **Verlet sphere radius**:
  $$r_v = r_c + \Delta$$
  where $\Delta$ is the **skin depth** (buffer thickness).
- **During force evaluation steps:** We loop *only* over the pre-stored neighbor table (averaging $m_v \approx \frac{4}{3}\pi (r_c + \Delta)^3 \rho$ entries per atom), checking whether $r_{ij} \le r_c$. This force evaluation step costs strictly **$\mathcal{O}(m_v N) = \mathcal{O}(N)$**!

### Exact Rebuild Criterion ($d_1 + d_2 \ge \Delta$)
Because particles move slowly per timestep ($\Delta r \sim 0.01\sigma$), a particle initially outside the Verlet sphere ($r_{ij}^{(0)} > r_c + \Delta$) cannot penetrate inside the force cutoff sphere ($r_{ij}(t) < r_c$) until the combined distance traveled by the pair since the last table build reaches $\Delta$.
- Let $\Delta\mathbf{r}_i = \mathbf{r}_i(t) - \mathbf{r}_i(t_{\text{build}})$ be the displacement of atom $i$ since the table was last constructed.
- Track the **two largest displacements** across the entire system: $d_1 = \max_i |\Delta\mathbf{r}_i|$ and $d_2 = \text{second largest } |\Delta\mathbf{r}_j|$.
- **Rebuild trigger:** Whenever
  $$d_1 + d_2 \ge \Delta \quad (\text{or conservatively } 2 d_{\max} \ge \Delta)$$
  we must reconstruct the neighbor table!

### The Skin Depth $\Delta$ Optimization Trade-off
- **If $\Delta$ is too small:** Very few excess particles lie in the skin ($m_v \approx m$), so each force step is fast, **but** $d_1 + d_2 \ge \Delta$ triggers almost every step! Since a pure Verlet table rebuild must check all pairs ($\mathcal{O}(N^2)$), frequent rebuilds ruin performance.
- **If $\Delta$ is too large:** The table rarely needs rebuilding ($n_{\text{reb}}$ is large), **but** the number of candidate pairs checked at every step grows cubically as $(r_c + \Delta)^3$.
- Modern MD codes dynamically optimize $\Delta$ by timing blocks of steps, fitting CPU time $t(\Delta)$ to a polynomial, and minimizing $t(\Delta)$.


In [25]:
ws.render_section('concept_verlet_skin')

### Interactive Explorer: Verlet Neighbor List Skin Depth Optimization

Vary the system size $N$ and reduced temperature $T^*$ to see how the per-step force evaluation cost $\propto N(r_c + \Delta)^3$ competes against the amortized $\mathcal{O}(N^2)$ list rebuild cost $\propto N^2 / n_{\mathrm{reb}}(\Delta)$.

**Key Takeaway:** There is a well-defined optimal skin depth $\Delta^*/r_c$ (typically $0.15\text{–}0.35$, rebuilding every $10\text{–}25$ steps). Increasing $N$ or $T^*$ makes $\mathcal{O}(N^2)$ rebuilds more costly per step, shifting $\Delta^*$ to larger values.

In [26]:
ws.render_section('q19_all_pairs_count')

**Q19 · easy** · 💡

In an all-pairs force calculation using Newton's third law (`j in range(i+1, N)`), how many distinct pair distance checks $\frac{1}{2}N(N-1)$ are performed per MD timestep for a system of $N = 4000$ atoms? (Enter the exact integer.)


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $4000 \times 3999 / 2$.

</div>
</details>

_Numeric answer required._

In [27]:
ws.render_section('q20_average_neighbors_independence')

**Q20 · easy** · 💡

Consider two bulk liquid Argon simulations at the same number density $\rho = 0.84\,\sigma^{-3}$ and cutoff $r_c = 2.5\,\sigma$: System A has $N = 1000$ atoms and System B has $N = 1,000,000$ atoms. How does the average number of interacting neighbors $m$ inside $r_c$ for a single atom compare between System A and System B?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall $m \approx \frac{4}{3}\pi r_c^3 \rho$.

</div>
</details>

- [ ] **(A)** $m$ is identical in both systems (approx. $55$ neighbors) because $m \approx \frac{4}{3}\pi r_c^3 \rho$ depends only on local density $\rho$ and $r_c$, not $N$
- [ ] **(B)** $m$ is $1000\times$ larger in System B because $N$ is $1000\times$ larger
- [ ] **(C)** $m$ is $10\times$ larger in System B because box length $L$ is $10\times$ larger
- [ ] **(D)** $m$ is smaller in System B because periodic boundary conditions dilute the density

In [28]:
ws.render_section('q21_verlet_rebuild_criterion')

**Q21 · medium** · 💡

In a Verlet neighbor list with force cutoff $r_c$ and skin depth $\Delta$ (outer list radius $r_v = r_c + \Delta$), let $d_1$ and $d_2$ be the two largest atomic displacements $|\mathbf{r}_i(t) - \mathbf{r}_i(t_{\text{build}})|$ accumulated **since the neighbor list was last built**. What is the exact condition that guarantees no unrecorded pair has entered $r < r_c$, and when must the list be rebuilt?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

In the worst case, two atoms initially just outside $r_c + \Delta$ move directly toward each other by distances $d_1$ and $d_2$.

</div>
</details>

- [ ] **(A)** Rebuild as soon as $d_1 + d_2 \ge \Delta$ (or conservatively $2 d_1 \ge \Delta$), measured relative to positions at the last rebuild
- [ ] **(B)** Rebuild only when $d_1 \ge r_c + \Delta$, measured over a single timestep $\Delta t$
- [ ] **(C)** Rebuild when the average displacement $\langle d_i \rangle \ge \Delta / 2$
- [ ] **(D)** Rebuild only when the total system energy drifts by more than $\Delta$

In [29]:
ws.render_section('q22_verlet_displacement_reference_bug')

**Q22 · hard** · 💡

A student implements a Verlet neighbor list rebuild check by computing `d_i = norm(r[i, t] - r[i, t - dt])` (the displacement during the *current single timestep* $\Delta t$) and rebuilding when `max1(d) + max2(d) >= Delta`. Why is this implementation **fundamentally buggy**?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

What happens if an atom moves by $0.05\,\Delta$ per step in the same direction for 15 consecutive steps?

</div>
</details>

- [ ] **(A)** Displacements accumulate over multiple timesteps; checking only the single-step displacement $\mathbf{r}(t) - \mathbf{r}(t-\Delta t)$ misses atoms that drift steadily across the skin $\Delta$ over several steps
- [ ] **(B)** Single-step displacements are always larger than multi-step displacements, causing rebuilds on every step
- [ ] **(C)** The rebuild check must use velocities $\mathbf{v}_i$ instead of positions
- [ ] **(D)** The skin depth $\Delta$ must be divided by $N$ when checking single-step displacements

In [30]:
ws.render_section('q23_verlet_skin_volume_ratio')

**Q23 · medium** · 💡

In a 3D simulation with force cutoff $r_c = 2.5\,\sigma$ and Verlet skin thickness $\Delta = 0.5\,\sigma$ (so $r_v = r_c + \Delta = 3.0\,\sigma$), by what factor $(r_v / r_c)^3$ does the volume of the Verlet sphere (and thus the average number of neighbors stored in the list per atom) exceed the actual force cutoff sphere? (Enter a number to 3 decimal places, e.g., `1.728`.)


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

In 3D, sphere volume scales as $r^3$, so compute $(3.0 / 2.5)^3 = 1.2^3$.

</div>
</details>

_Numeric answer required._

In [31]:
ws.render_section('q24_verlet_skin_zero_limit')

**Q24 · medium** · 💡

What happens to the computational efficiency of a Verlet neighbor list algorithm if you set the skin depth to $\Delta = 0$ ($r_v = r_c$)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

If $\Delta = 0$, how many timesteps can the table be reused before $d_1 + d_2 \ge 0$?

</div>
</details>

- [ ] **(A)** The table must be rebuilt at every single timestep ($\mathcal{O}(N^2)$ every step), making it slower than having no neighbor list at all due to table-construction overhead
- [ ] **(B)** It achieves the theoretical minimum CPU time because no extra pairs in the skin are ever checked
- [ ] **(C)** The simulation remains $\mathcal{O}(N)$ per step with zero memory overhead
- [ ] **(D)** Particles are frozen in place because $d_1 + d_2 \le 0$

In [32]:
ws.render_section('q25_verlet_force_eval_check')

**Q25 · easy** · 💡

During a regular MD timestep (between list rebuilds), when iterating over the pairs $(i, j)$ stored in the Verlet neighbor table (built with radius $r_v = r_c + \Delta$), must the code still check `if r2 <= rcutoff2` before computing the pair force?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Remember that the Verlet table contains all neighbors up to $r_v = r_c + \Delta$, not just $r_c$.

</div>
</details>

- [ ] **(A)** Yes, because the Verlet table includes buffer particles in the skin shell $r_c < r_{ij} \le r_c + \Delta$ whose interaction potential is zero
- [ ] **(B)** No, all particles in the Verlet neighbor table are guaranteed to have $r_{ij} \le r_c$
- [ ] **(C)** No, because particles in the skin $r_c < r_{ij} \le r_v$ should also exert full Lennard-Jones forces
- [ ] **(D)** Only on steps when the neighbor table is being rebuilt

In [33]:
ws.render_section('q26_pure_verlet_asymptotic_complexity')

**Q26 · medium** · 💡

If a Verlet neighbor list is reconstructed every $n_{\text{reb}} = 15$ steps using a standard all-pairs loop (without cell binning), what is the formal asymptotic complexity per step as $N \to \infty$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Amortize the $\mathcal{O}(N^2)$ rebuild cost over $n_{\text{reb}}$ steps: $c_1 N + \frac{c_2}{n_{\text{reb}}} N^2$.

</div>
</details>

- [ ] **(A)** Still $\mathcal{O}(N^2)$ asymptotically (with a prefactor reduced by $1/n_{\text{reb}}$), because the rebuild step itself checks all $N(N-1)/2$ pairs
- [ ] **(B)** Strictly $\mathcal{O}(N)$ for all $N \to \infty$
- [ ] **(C)** Strictly $\mathcal{O}(\log N)$
- [ ] **(D)** $\mathcal{O}(N^{3/2})$

In [34]:
ws.render_section('q27_verlet_memory_scaling')

**Q27 · easy** · 💡

As shown on slide 8 & 10 of `05.PBC-neighbors.pdf`, storing an explicit Verlet neighbor table for $N$ atoms where each atom has on average $m_v$ neighbors in its list requires memory scaling as:


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Each of the $N$ columns/rows in the neighbor table stores up to $\sim m_v$ neighbor indices.

</div>
</details>

- [ ] **(A)** $\mathcal{O}(m_v N)$ integers
- [ ] **(B)** $\mathcal{O}(N^2)$ integers
- [ ] **(C)** $\mathcal{O}(1)$ integers
- [ ] **(D)** $\mathcal{O}(m_v^3)$ integers

In [35]:
ws.render_section('q28_steps_between_rebuilds_calc')

**Q28 · medium** · 💡

In a liquid MD simulation with timestep $\Delta t = 0.005\,\tau$ and skin depth $\Delta = 0.40\,\sigma$, the fastest atoms move at $v_{\max} = 4.0\,\sigma/\tau$. Assuming the worst-case bound where the two fastest atoms travel straight lines ($d_1 = d_2 = v_{\max} n \Delta t$), after how many timesteps $n$ will the rebuild condition $d_1 + d_2 \ge \Delta$ be triggered?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Solve $2 v_{\max} n \Delta t = \Delta$ for $n$.

</div>
</details>

_Numeric answer required._

In [36]:
ws.render_section('sec_cell_linked_lists')

## 4. The Cell Linked-List (Bin Sort) Method & Hybrid $\mathcal{O}(N)$ Synergy

To eliminate the $\mathcal{O}(N^2)$ cost of constructing neighbor tables, we partition the simulation box of side $L$ into a regular $M \times M \times M$ grid of cubic **cells (bins)** of side length:
$$l_c = \frac{L}{M} \ge r_c \quad (\text{or } l_c \ge r_c + \Delta \text{ when building a Verlet list})$$
where $M = \lfloor L / r_c \rfloor$.

### Why is the Cell Method Strictly $\mathcal{O}(N)$?
1. **Binning particles ($\mathcal{O}(N)$):** Each particle $i$ is assigned to its cell index in $\mathcal{O}(1)$ arithmetic operations via `cx = int(x[i] / lc)`, `cy = int(y[i] / lc)`, `cz = int(z[i] / lc)`.
2. **Local cell stencil search ($\mathcal{O}(N)$):** Because $l_c \ge r_c$, any particle within distance $r_c$ of atom $i$ **must reside either in atom $i$'s own cell or in one of the immediately adjacent cells**:
   - In **2D**: $3^2 = \mathbf{9}$ cells total (or **5 cells** using Newton's 3rd Law half-stencil).
   - In **3D**: $3^3 = \mathbf{27}$ cells total (or $1 + \frac{26}{2} = \mathbf{14}$ cells using Newton's 3rd Law half-stencil).
   Since each cell contains a constant average number of particles $N_c = N / M^3 \approx \rho r_c^3$, checking 27 cells per particle takes $\mathcal{O}(1)$ work per particle $\implies$ **strictly $\mathcal{O}(N)$ total time**!

### The `head[c]` and `list[i]` Linked-List Data Structure
Instead of allocating variable-length 2D arrays for every cell, MD codes represent all cell contents using just **two 1D static integer arrays** (with `-1` denoting the end-of-list sentinel):
- `head[c]` (length $M^d$): stores the index of the **first** (most recently inserted) particle in cell `c`, or `-1` if cell `c` is empty.
- `list[i]` (length $N$): stores the index of the **next** particle in the same cell as particle `i`, or `-1` if `i` is the last particle in that cell.

**$\mathcal{O}(N)$ Construction Loop:**
```python
head[:] = -1
for i in range(N):
    c = cell_index(r[i])
    list[i] = head[c]   # point i to previous head of cell c
    head[c] = i         # make i the new head of cell c
```
- **Memory footprint:** Requires only $M^d + N = \mathcal{O}(N)$ integers (rather than $\mathcal{O}(mN)$ for an explicit Verlet table!), with zero dynamic memory allocation.

### Hybrid Cell + Verlet Neighbor List (Best of Both Worlds)
Why do state-of-the-art MD codes (like LAMMPS and GROMACS) use **Cell Linked-Lists to build a Verlet Neighbor List** every $10\text{–}20$ steps, rather than evaluating forces directly from the 27-cell stencil at every step?
- Compare the **search volume** per atom:
  - 27 cubic cells of side $l_c = r_c$ have volume $V_{\text{cells}} = 27 r_c^3$.
  - A spherical cutoff (or Verlet sphere) has volume $V_{\text{sphere}} = \frac{4}{3}\pi r_c^3 \approx 4.189 r_c^3$.
  - Ratio:
    $$\frac{V_{\text{cells}}}{V_{\text{sphere}}} = \frac{27 r_c^3}{\frac{4}{3}\pi r_c^3} = \frac{81}{4\pi} \approx 6.45$$
- Directly scanning 27 cells checks **$6.45\times$ more candidate pairs** than actually lie inside the sphere $r_c$! By using the $\mathcal{O}(N)$ Cell Linked-List only once every $\sim 15$ steps to populate a tight spherical Verlet list of radius $r_c + \Delta$, both the rebuild step and the force step are **strictly $\mathcal{O}(N)$** *and* we only check $\sim (1 + \Delta/r_c)^3 \approx 1.7\times$ the sphere volume on intermediate steps!


In [37]:
ws.render_section('concept_cell_linked_list')

### Interactive Explorer: 2D Cell Linked-List Pointer Chain & Stencil

Select any cell $c \in \{0..8\}$ in the $3 \times 3$ periodic grid and toggle between the full 9-cell stencil and Newton's 3rd-law 5-cell half-stencil. Trace how `head[c]` and `list[i]` chain particles inside each cell using only 1D integer arrays.

**Key Takeaway:** When particles `5`, `6`, and `7` are inserted sequentially into Cell 4 (`list[i] = head[4]; head[4] = i`), `head[4]` holds `7`, `list[7] = 6`, `list[6] = 5`, and `list[5] = -1` (LIFO order).

In [38]:
ws.render_section('q29_cell_size_condition')

**Q29 · easy** · 💡

In the Cell Method (when used directly for force evaluation with cutoff $r_c$, or when building a Verlet list with cutoff $r_v = r_c + \Delta$), what condition must the cell edge length $l_c = L / M$ satisfy so that we only need to search a particle's own cell and immediately adjacent cells?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

If a neighbor cell is separated by one full cell width $l_c$ between non-adjacent cells, we need $l_c$ to be at least the interaction/list cutoff.

</div>
</details>

- [ ] **(A)** $l_c \ge r_c$ (or $l_c \ge r_c + \Delta$ when constructing a Verlet list), so $M = \lfloor L / r_c \rfloor$
- [ ] **(B)** $l_c \le r_c / 4$
- [ ] **(C)** $l_c = L / N$
- [ ] **(D)** $l_c > L / 2$

In [39]:
ws.render_section('q30_max_cells_1d_calc')

**Q30 · easy** · 💡

A 3D cubic periodic simulation box has side length $L = 22.0\,\sigma$, and you use the Cell Method to build a Verlet neighbor list of radius $r_v = r_c + \Delta = 3.0\,\sigma$. What is the **maximum number of cells per edge $M$** (where $l_c = L / M \ge r_v$) that you can partition the box into along each axis?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $M = \lfloor L / r_v \rfloor = \lfloor 22.0 / 3.0 \rfloor$.

</div>
</details>

_Numeric answer required._

In [40]:
ws.render_section('q31_3d_stencil_counts')

**Q31 · easy** · 💡

In a **3D** Cell Linked-List simulation ($M \ge 3$), how many cells (including the central cell itself) must be searched for each cell without Newton's 3rd Law, and how many when exploiting Newton's 3rd Law ($\mathbf{F}_{ij} = -\mathbf{F}_{ji}$) to avoid double-checking pairs of cells?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

In 3D, a $3 \times 3 \times 3$ cube has $27$ cells: $1$ central cell plus $26 / 2 = 13$ forward neighbor cells.

</div>
</details>

- [ ] **(A)** 27 cells total without Newton's 3rd Law; 14 cells (1 self + 13 forward neighbors) with Newton's 3rd Law
- [ ] **(B)** 9 cells total without Newton's 3rd Law; 5 cells with Newton's 3rd Law
- [ ] **(C)** 6 cells total without Newton's 3rd Law; 3 cells with Newton's 3rd Law
- [ ] **(D)** 64 cells total without Newton's 3rd Law; 32 cells with Newton's 3rd Law

In [41]:
ws.render_section('q32_linked_list_trace_head')

**Q32 · medium** · 💡

Suppose `head` is initialized to `-1` for all cells. While looping `i = 0, 1, ..., 9` in ascending order, particles `i = 2`, `i = 5`, and `i = 8` are the only particles that map to cell `c = 3` using:
```python
list[i] = head[c]
head[c] = i
```
After the loop finishes, what integer value is stored in `head[3]`?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Each insertion prepends the new particle to the front (head) of the linked list (LIFO order).

</div>
</details>

_Numeric answer required._

In [42]:
ws.render_section('q33_linked_list_trace_next')

**Q33 · medium** · 💡

In the same scenario as the previous question (particles `2`, `5`, and `8` inserted sequentially into cell `c = 3`), what integer value is stored in `list[5]` after the construction loop finishes?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

When `i = 5` was inserted, what was the value of `head[3]` right before `head[3]` was overwritten with `5`?

</div>
</details>

_Numeric answer required._

In [43]:
ws.render_section('q34_linked_list_memory_advantage')

**Q34 · easy** · 💡

According to slide 10 of `05.PBC-neighbors.pdf` ("Improvements: the Cell Method"), how does the memory requirement of the Cell Linked-List method compare to an explicit Verlet neighbor table?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Count the size of `list[0..N-1]` plus `head[0..M^3-1]` versus a table storing $m$ neighbors for each of the $N$ atoms.

</div>
</details>

- [ ] **(A)** Cell Linked-List memory scales as $\mathcal{O}(N)$ (specifically $N + M^3$ integers), whereas an explicit neighbor table requires $\mathcal{O}(mN)$ memory
- [ ] **(B)** Cell Linked-List memory scales as $\mathcal{O}(N^2)$, whereas an explicit neighbor table requires $\mathcal{O}(N)$
- [ ] **(C)** Both require $\mathcal{O}(N^2)$ floating-point arrays
- [ ] **(D)** Cell Linked-List requires dynamic heap allocations (`malloc`) for every pair interaction

In [44]:
ws.render_section('q35_cell_vs_sphere_volume_ratio')

**Q35 · hard** · 💡

In 3D, suppose the cell size equals the cutoff radius ($l_c = r_c$). The 27 neighbor cells searched around an atom have total volume $V_{\text{cells}} = 27 r_c^3$, whereas the actual interaction sphere has volume $V_{\text{sphere}} = \frac{4}{3}\pi r_c^3$. Calculate the ratio $V_{\text{cells}} / V_{\text{sphere}} = \frac{81}{4\pi}$ to two decimal places (e.g., `6.45`).


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Divide $27$ by $(4\pi / 3) \approx 4.18879$.

</div>
</details>

_Numeric answer required._

In [45]:
ws.render_section('q36_hybrid_verlet_cell_synergy')

**Q36 · medium** · 💡

Why do production MD codes (such as LAMMPS) combine **both** the Cell Linked-List method and a Verlet Neighbor List (using cells of size $l_c \ge r_c + \Delta$ to rebuild the Verlet list when $d_1 + d_2 \ge \Delta$)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Combine the $\mathcal{O}(N)$ rebuild scaling of Cell lists with the smaller spherical volume of Verlet lists on intermediate steps.

</div>
</details>

- [ ] **(A)** The Cell method makes neighbor list rebuilds strictly $\mathcal{O}(N)$ instead of $\mathcal{O}(N^2)$, while the spherical Verlet list avoids checking the $\sim 6.45\times$ excess volume of the 27 cubic cells on every intermediate timestep
- [ ] **(B)** The Cell method alone is $\mathcal{O}(N^3)$, so it needs a Verlet list to become $\mathcal{O}(N^2)$
- [ ] **(C)** Combining them allows the cutoff $r_c$ to exceed the box length $L$
- [ ] **(D)** The Verlet list eliminates the need for Periodic Boundary Conditions

In [46]:
ws.render_section('q37_when_cell_method_fails_small_box')

**Q37 · medium** · 💡

In a small simulation where the box length is $L = 6.0\,\sigma$ and the cutoff is $r_c = 2.5\,\sigma$, what happens when you attempt to partition the box into $M = \lfloor L / r_c \rfloor$ cells per axis for the 27-cell stencil method?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Calculate $M = \lfloor 6.0 / 2.5 \rfloor$. How many cells exist in a $3 \times 3 \times 3$ stencil if $M < 3$?

</div>
</details>

- [ ] **(A)** $M = \lfloor 6.0 / 2.5 \rfloor = 2 < 3$, so a $3 \times 3 \times 3$ neighbor stencil wraps around and visits the same cells multiple times; the standard 27-cell stencil requires $M \ge 3$ ($L \ge 3 r_c$)
- [ ] **(B)** The Minimum Image Convention $r_c < L/2$ is violated because $2.5 > 6.0 / 2$
- [ ] **(C)** The `head` array requires infinite memory when $M = 2$
- [ ] **(D)** The cell method works without modification for any $M \ge 1$

In [47]:
ws.render_section('q38_complexity_summary_comparison')

**Q38 · easy** · 💡

Which of the following correctly summarizes the asymptotic time complexity of (1) Naive all-pairs force evaluation, (2) Force evaluation using a pre-built Verlet neighbor table, and (3) Neighbor table construction using the Cell Linked-List method, for a short-range potential at fixed density $\rho$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Review slides 6, 7, and 10 of `05.PBC-neighbors.pdf`.

</div>
</details>

- [ ] **(A)** (1) All-pairs: $\mathcal{O}(N^2)$ · (2) Force step with neighbor table: $\mathcal{O}(N)$ · (3) Cell Linked-List construction: $\mathcal{O}(N)$
- [ ] **(B)** (1) All-pairs: $\mathcal{O}(N^2)$ · (2) Force step with neighbor table: $\mathcal{O}(N \log N)$ · (3) Cell Linked-List construction: $\mathcal{O}(N^2)$
- [ ] **(C)** (1) All-pairs: $\mathcal{O}(N^{3/2})$ · (2) Force step with neighbor table: $\mathcal{O}(N)$ · (3) Cell Linked-List construction: $\mathcal{O}(N \log N)$
- [ ] **(D)** (1) All-pairs: $\mathcal{O}(N^3)$ · (2) Force step with neighbor table: $\mathcal{O}(N^2)$ · (3) Cell Linked-List construction: $\mathcal{O}(N)$

In [ ]:
ws.show_record()

In [ ]:
ws.finalize()